# 🎵 AutoLyrics — DALI Dataset Pipeline

**Stage 2 Notebook — Multi-Sample DALI Processing + Lyrics Formatting**

---

### What this notebook does:
1. Installs dependencies and loads the Whisper model
2. Loads the **DALI dataset** (multiple songs with ground-truth lyrics)
3. For each sample: preprocesses audio → chunks → transcribes with timestamps
4. Merges all chunk outputs and deduplicates overlapping segments
5. Formats output into **proper lyrics** (lines + verse breaks) using both
   - DALI's ground-truth line-level timing (preferred, when available)
   - Pause-gap heuristics (fallback)
6. Computes WER/CER vs. ground truth for every song
7. Saves per-song `.txt` files and a summary report

### Model used: `openai/whisper-base`
- Better accuracy than Whisper Tiny
- Fast on Colab T4 GPU
- Drop-in upgrade path to `whisper-small` / `whisper-medium`

> **Tip:** Go to `Runtime → Change runtime type → T4 GPU` before running.

---
## Section 1 — Setup

Install all required libraries.

In [ ]:
!pip install -q transformers torchaudio librosa soundfile
!pip install -q jiwer datasets
# DALI Python API — install from the official repo
!pip install -q DALI-dataset
# Note: if DALI-dataset install fails, the notebook falls back to the
# Hugging Face Hub version of DALI loaded via `datasets` (see Section 2B).

In [ ]:
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
import textwrap
import os
import re
import json
from pathlib import Path

from transformers import WhisperProcessor, WhisperForConditionalGeneration
from IPython.display import Audio, display
from jiwer import wer, cer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

---
## Section 2 — Model Loading

In [ ]:
MODEL_NAME = "openai/whisper-base"

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model     = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model     = model.to(device)
model.eval()

print(f"✅ Model '{MODEL_NAME}' loaded on {device}")

---
## Section 3 — DALI Dataset Loading

DALI provides:
- Raw audio (`.wav`) per song
- Ground-truth lyrics at **note**, **word**, and **line** level
- Timestamps for each annotation level

We use **line-level** timestamps to know exactly where each lyric line
starts and ends — much more accurate than heuristic pause detection.

### Option A — DALI Python API (recommended if you have the dataset files)
```
DALI_PATH = "/path/to/DALI_v2.0"
AUDIO_PATH = "/path/to/DALI_v2.0/audio"
```

### Option B — Hugging Face Hub (easier, no local files needed)
The dataset is `dali-dataset/DALI` on Hugging Face Hub.
Some songs may not include audio due to licensing; in that case the
notebook will skip those entries automatically.

**Set `USE_HF_DATASET = True` below if you do not have local DALI files.**

In [ ]:
# ============================================================
#  DATASET CONFIGURATION — adjust these paths / flags
# ============================================================

# Set to True  → load from Hugging Face Hub (no local files needed)
# Set to False → use local DALI files at DALI_DATA_PATH
USE_HF_DATASET = True

# --- Local DALI paths (only needed if USE_HF_DATASET = False) ---
DALI_DATA_PATH  = "/content/DALI_v2.0"      # folder with DALI .gz annotation files
DALI_AUDIO_PATH = "/content/DALI_v2.0/audio" # folder with per-song .wav files

# How many songs to process (set to None to process all)
MAX_SAMPLES = 10

# Where to save per-song output lyrics
OUTPUT_DIR = "/content/autolyrics_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Language & task for Whisper
LANGUAGE = "en"
TASK     = "transcribe"
# ============================================================

In [ ]:
# ────────────────────────────────────────────────────────────
# Helper: normalise a DALI entry into a standard dict:
#   {
#     'id'         : str,
#     'title'      : str,
#     'audio_path' : str | None,   # path to .wav on disk
#     'lines'      : list of {'text': str, 'time': [start, end]}
#   }
# ────────────────────────────────────────────────────────────

def load_dali_local(dali_data_path, dali_audio_path, max_samples=None):
    """Load DALI using the official DALI Python API."""
    import DALI as dali_api

    print("Loading DALI annotations from disk …")
    dali_data = dali_api.get_the_DALI_dataset(
        dali_data_path,
        skip=[], keep=[]
    )

    samples = []
    for song_id, entry in dali_data.items():
        if max_samples and len(samples) >= max_samples:
            break

        audio_file = os.path.join(dali_audio_path, f"{song_id}.wav")
        if not os.path.isfile(audio_file):
            # Try .mp3
            audio_file_mp3 = os.path.join(dali_audio_path, f"{song_id}.mp3")
            audio_file = audio_file_mp3 if os.path.isfile(audio_file_mp3) else None

        # Extract line-level annotations
        lines = []
        try:
            for line in entry.annotations["annot"]["lines"]:
                lines.append({
                    "text": line["text"],
                    "time": line["time"]   # [start_sec, end_sec]
                })
        except (KeyError, TypeError):
            pass  # annotation format may vary between DALI versions

        samples.append({
            "id"         : song_id,
            "title"      : getattr(entry.info, "title", song_id),
            "audio_path" : audio_file,
            "lines"      : lines
        })

    print(f"   Loaded {len(samples)} entries from local DALI.")
    return samples


def load_dali_hf(max_samples=None):
    """Load DALI via the Hugging Face Hub."""
    from datasets import load_dataset

    print("Streaming DALI from Hugging Face Hub …")
    # 'dali-dataset/DALI' — english split; streaming avoids a large download
    hf_ds = load_dataset(
        "dali-dataset/DALI",
        split="train",
        streaming=True,
        trust_remote_code=True
    )

    samples = []
    for row in hf_ds:
        if max_samples and len(samples) >= max_samples:
            break

        # Save audio to a temp file so librosa can load it
        audio_path = None
        if row.get("audio") is not None:
            tmp_path = f"/tmp/dali_{row['id']}.wav"
            audio_arr = np.array(row["audio"]["array"], dtype=np.float32)
            sr_orig   = row["audio"]["sampling_rate"]
            sf.write(tmp_path, audio_arr, sr_orig)
            audio_path = tmp_path

        # Parse line annotations from the HF schema
        lines = []
        for line in row.get("lines", []):
            lines.append({
                "text": line["text"],
                "time": [line["start"], line["end"]]
            })

        samples.append({
            "id"         : row["id"],
            "title"      : row.get("title", row["id"]),
            "audio_path" : audio_path,
            "lines"      : lines
        })

    print(f"   Loaded {len(samples)} entries from Hugging Face DALI.")
    return samples


# ─── Load the dataset ───────────────────────────────────────
if USE_HF_DATASET:
    dali_samples = load_dali_hf(max_samples=MAX_SAMPLES)
else:
    dali_samples = load_dali_local(DALI_DATA_PATH, DALI_AUDIO_PATH, max_samples=MAX_SAMPLES)

print(f"\n✅ Dataset ready — {len(dali_samples)} songs to process.")

---
## Section 4 — Audio Chunking

Whisper's native window is 30 seconds. We split each song into overlapping
chunks so no word is cut at a boundary.

In [ ]:
CHUNK_DURATION_SEC = 30
OVERLAP_SEC        = 5
TARGET_SAMPLE_RATE = 16000  # Hz — Whisper requires 16kHz

In [ ]:
def load_and_resample_audio(audio_path, target_sr=16000):
    """
    Load any audio file (wav/mp3/flac …) and resample to target_sr.
    Returns (audio_array, sample_rate).
    """
    audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    return audio, sr


def split_audio_into_chunks(audio_array, sample_rate,
                             chunk_sec=30, overlap_sec=5):
    """
    Split audio into overlapping chunks.

    Returns list of (chunk_array, chunk_start_time_sec).
    """
    chunk_samples   = chunk_sec   * sample_rate
    overlap_samples = overlap_sec * sample_rate
    step_samples    = chunk_samples - overlap_samples

    chunks = []
    start  = 0

    while start < len(audio_array):
        end        = min(start + chunk_samples, len(audio_array))
        chunk      = audio_array[start:end]
        start_time = start / sample_rate

        chunks.append((chunk, start_time))
        start += step_samples
        if end == len(audio_array):
            break

    return chunks

---
## Section 5 — Feature Extraction

In [ ]:
def extract_features(chunk_audio, processor, sample_rate=16000):
    """
    Convert a raw audio array into Whisper input features (log-Mel spectrogram).
    """
    inputs = processor(
        chunk_audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )
    return inputs.input_features.to(device)

---
## Section 6 — Whisper Inference

In [ ]:
def parse_whisper_timestamps(token_string):
    """
    Parse timestamp tokens (<|0.00|>, <|2.50|>, …) out of a Whisper
    decoded string and return a list of segment dicts:
      {'text': str, 'timestamp': (start_sec, end_sec)}
    """
    pattern = r"<\|(\d+\.\d+)\|>"
    parts   = re.split(pattern, token_string)

    segments = []
    i = 0

    # Skip leading non-timestamp content
    while i < len(parts) and not re.fullmatch(r"\d+\.\d+", parts[i].strip()):
        i += 1

    while i < len(parts) - 1:
        try:
            start_time = float(parts[i])
        except ValueError:
            i += 1
            continue

        raw_text = parts[i + 1] if i + 1 < len(parts) else ""
        raw_text = re.sub(r"<\|[^>]+\|>", "", raw_text).strip()

        # ── FIX: use the REAL end timestamp, not a hardcoded +2.0 ──
        try:
            end_time = float(parts[i + 2]) if i + 2 < len(parts) else start_time + 2.0
        except ValueError:
            end_time = start_time + 2.0

        if raw_text:
            segments.append({
                "text"      : raw_text,
                "timestamp" : (start_time, end_time)   # real end time
            })

        i += 2

    return segments


def transcribe_chunk(input_features, model, processor,
                     language="en", task="transcribe"):
    """
    Run Whisper on one chunk's features.
    Returns list of dicts: {'text', 'timestamp': (start, end)}
    """
    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=language, task=task
    )

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            return_timestamps=True
        )

    raw_decoded = processor.tokenizer.decode(
        predicted_ids[0].tolist(),
        skip_special_tokens=False
    )

    segments = parse_whisper_timestamps(raw_decoded)

    if not segments:   # fallback — no timestamps parsed
        plain = processor.tokenizer.decode(
            predicted_ids[0].tolist(), skip_special_tokens=True
        ).strip()
        if plain:
            segments = [{"text": plain, "timestamp": (0.0, 30.0)}]

    return segments


def transcribe_audio(audio_array, sample_rate,
                     chunk_sec=30, overlap_sec=5,
                     language="en", task="transcribe"):
    """
    Full pipeline: split → extract → infer → merge → return all segments.

    Returns:
        List of dicts: {'text', 'timestamp': (start, end), 'abs_timestamp': (start, end)}
    """
    chunks      = split_audio_into_chunks(audio_array, sample_rate, chunk_sec, overlap_sec)
    all_segments = []

    for idx, (chunk_audio, chunk_start) in enumerate(chunks):
        features = extract_features(chunk_audio, processor, sample_rate)
        segs     = transcribe_chunk(features, model, processor, language, task)

        for seg in segs:
            ts = seg["timestamp"]
            # ── FIX: propagate the REAL end time into abs_timestamp ──
            abs_start = chunk_start + ts[0]
            abs_end   = chunk_start + ts[1]   # ts[1] is the true end, not +2.0
            seg["abs_timestamp"] = (abs_start, abs_end)
            all_segments.append(seg)

    return all_segments

---
## Section 7 — Deduplication & Segment Cleaning

In [ ]:
def clean_segment_text(text):
    """Strip whitespace, collapse internal spaces."""
    text = text.strip()
    text = " ".join(text.split())
    return text


def should_skip_segment(text):
    """
    Return True if the segment is too short or looks like noise.
    This filter is applied BEFORE adding a segment to the cleaned list.
    """
    # Skip if text is under 2 characters after cleaning
    if len(text.strip()) < 2:
        return True
    # Skip common transcription noise artefacts
    noise_phrases = {"[music]", "[applause]", "[noise]", "...", "…"}
    if text.strip().lower() in noise_phrases:
        return True
    return False


def deduplicate_and_clean_segments(segments, time_tolerance=2.0):
    """
    Remove duplicates caused by chunk overlap, clean text, and return
    a sorted list of (start_time, end_time, text) triples.

    Two segments are duplicates if their text matches AND their start
    times are within `time_tolerance` seconds.

    NOTE: we now carry the real end_time through so the formatter
          can compute accurate gaps between segments.
    """
    raw = []
    for seg in segments:
        text = clean_segment_text(seg.get("text", ""))

        # ── FIX: call should_skip_segment (was defined but never used) ──
        if should_skip_segment(text):
            continue

        start, end = seg["abs_timestamp"]
        raw.append((start, end, text))

    # Sort by start time
    raw.sort(key=lambda x: x[0])

    # Deduplicate
    cleaned    = []
    seen_texts = {}   # text → last start time we saw it at

    for (start, end, text) in raw:
        last_seen = seen_texts.get(text, -999)
        if (start - last_seen) > time_tolerance:
            cleaned.append((start, end, text))
            seen_texts[text] = start

    return cleaned   # list of (start, end, text)

---
## Section 8 — Lyrics Formatting

### Strategy A — DALI ground-truth line boundaries (preferred)
DALI tells us exactly when each lyric line starts and ends.
We use those boundaries to decide line breaks instead of guessing from
audio gaps.  This gives lyrics that look like real song lyrics.

### Strategy B — Pause-gap heuristics (fallback)
When DALI line annotations are not available we fall back to the
time-gap approach from the original notebook — with the key fix that
we now use the **real** end timestamp (not a fixed +2.0 s approximation).

In [ ]:
# ============================================================
#  LYRICS FORMATTING CONFIGURATION
# ============================================================

# Gap thresholds for Strategy B (heuristic)
NEW_LINE_GAP_SEC   = 1.5   # gap ≥ this  → new line
NEW_VERSE_GAP_SEC  = 3.5   # gap ≥ this  → new verse (blank line)
MAX_LINE_LENGTH    = 60    # characters before word-wrap
# ============================================================

In [ ]:
def capitalize_line(text):
    """Capitalise the first character of a lyrics line."""
    if not text:
        return text
    return text[0].upper() + text[1:]


def wrap_long_line(line, max_len=60):
    """
    If `line` exceeds max_len, break it at word boundaries.
    Returns a list of sub-lines (each already capitalised).
    """
    if len(line) <= max_len:
        return [line]
    return [capitalize_line(w) for w in textwrap.wrap(line, width=max_len)]


# ──────────────────────────────────────────────────────────
#  Strategy A: use DALI ground-truth line boundaries
# ──────────────────────────────────────────────────────────

def format_lyrics_with_dali_lines(cleaned_segments, dali_lines,
                                  max_line_len=60):
    """
    Assign each transcribed segment to the DALI lyric line whose time
    range it falls into, then emit one output line per DALI line.

    A blank line is inserted between DALI lines whose gap > NEW_VERSE_GAP_SEC.

    Args:
        cleaned_segments : List of (start, end, text) from deduplicate_and_clean_segments
        dali_lines       : List of {'text': str, 'time': [start, end]} from DALI
        max_line_len     : Max chars per output line

    Returns:
        List of strings (lyrics lines, blank lines for verse breaks)
    """
    if not dali_lines:
        # No DALI annotations → fall back to heuristic
        return format_lyrics_heuristic(cleaned_segments, max_line_len=max_line_len)

    # Build a bucket for each DALI line
    buckets = [[] for _ in dali_lines]  # buckets[i] = list of segment texts

    for seg_start, seg_end, seg_text in cleaned_segments:
        seg_mid = (seg_start + seg_end) / 2   # use midpoint for assignment
        best_i  = None
        best_overlap = 0

        for i, dline in enumerate(dali_lines):
            ls, le = dline["time"][0], dline["time"][1]
            # Compute overlap between segment and DALI line window
            overlap = max(0, min(seg_end, le) - max(seg_start, ls))
            if overlap > best_overlap:
                best_overlap = overlap
                best_i = i

        # Fallback: assign by midpoint proximity if no overlap found
        if best_i is None:
            dists = [abs((seg_mid) - (d["time"][0] + d["time"][1]) / 2)
                     for d in dali_lines]
            best_i = int(np.argmin(dists))

        buckets[best_i].append(seg_text)

    # Render output lines
    output_lines = []
    prev_end = None

    for i, (dline, texts) in enumerate(zip(dali_lines, buckets)):
        ls, le = dline["time"][0], dline["time"][1]

        # Insert blank line for verse break
        if prev_end is not None and (ls - prev_end) >= NEW_VERSE_GAP_SEC:
            output_lines.append("")

        # Use transcribed text if available; fall back to ground-truth text
        if texts:
            line_text = " ".join(texts)
        else:
            line_text = dline["text"]   # ground-truth fallback

        line_text = capitalize_line(line_text.strip())
        output_lines.extend(wrap_long_line(line_text, max_line_len))

        prev_end = le

    return output_lines


# ──────────────────────────────────────────────────────────
#  Strategy B: pause-gap heuristic (fallback)
# ──────────────────────────────────────────────────────────

def format_lyrics_heuristic(cleaned_segments,
                             new_line_gap=NEW_LINE_GAP_SEC,
                             new_verse_gap=NEW_VERSE_GAP_SEC,
                             max_line_len=MAX_LINE_LENGTH):
    """
    Format lyrics using time gaps between segments.

    KEY FIX vs. original: `last_end_time` now comes from the REAL
    segment end timestamp (abs_timestamp[1]) instead of start + 2.0,
    so gap calculations are accurate.

    Args:
        cleaned_segments : List of (start, end, text)  ← now carries real end
    """
    if not cleaned_segments:
        return ["[No lyrics detected]"]

    lyrics_lines  = []
    current_line  = ""
    # ── FIX: initialise from the real end of segment 0, not start+2 ──
    last_end_time = cleaned_segments[0][1]   # index 1 = real end time

    for i, (start_time, end_time, text) in enumerate(cleaned_segments):

        gap = start_time - last_end_time

        if i == 0:
            current_line = text

        elif gap >= new_verse_gap:
            if current_line:
                lyrics_lines.append(capitalize_line(current_line))
            lyrics_lines.append("")   # verse break
            current_line = text

        elif gap >= new_line_gap:
            if current_line:
                lyrics_lines.append(capitalize_line(current_line))
            current_line = text

        else:
            current_line = (current_line + " " + text).strip()

        # ── FIX: use REAL end time ──
        last_end_time = end_time

    if current_line:
        lyrics_lines.append(capitalize_line(current_line))

    # Word-wrap long lines
    wrapped = []
    for line in lyrics_lines:
        if line == "":
            wrapped.append("")
        else:
            wrapped.extend(wrap_long_line(line, max_line_len))

    return wrapped


# ──────────────────────────────────────────────────────────
#  Dispatcher: pick Strategy A if DALI lines exist
# ──────────────────────────────────────────────────────────

def format_lyrics(cleaned_segments, dali_lines=None, max_line_len=MAX_LINE_LENGTH):
    """
    Route to the best available formatting strategy.
    """
    if dali_lines:
        return format_lyrics_with_dali_lines(cleaned_segments, dali_lines, max_line_len)
    else:
        return format_lyrics_heuristic(cleaned_segments, max_line_len=max_line_len)

---
## Section 9 — Evaluation (WER / CER)

We compare each song's transcription against the DALI ground-truth lyrics.

In [ ]:
def compute_metrics(predicted_lines, dali_lines):
    """
    Compute WER and CER between the transcribed output and DALI ground truth.

    Args:
        predicted_lines : list of strings (output of format_lyrics)
        dali_lines      : list of {'text', 'time'} dicts

    Returns:
        {'wer': float, 'cer': float}  (values in [0, 1])
    """
    hypothesis = " ".join(l for l in predicted_lines if l).lower().strip()
    reference  = " ".join(d["text"] for d in dali_lines).lower().strip()

    if not reference or not hypothesis:
        return {"wer": None, "cer": None}

    return {
        "wer": round(wer(reference, hypothesis), 4),
        "cer": round(cer(reference, hypothesis), 4)
    }

---
## Section 10 — Main Loop: Process All DALI Samples

This is the core of the multi-sample pipeline. For each song:
1. Load and resample audio
2. Transcribe all chunks
3. Clean and deduplicate segments
4. Format into lyrics (using DALI line boundaries when available)
5. Evaluate WER/CER
6. Save to disk

In [ ]:
results_summary = []   # collects {id, title, wer, cer} for the report

for sample_idx, sample in enumerate(dali_samples):
    song_id    = sample["id"]
    title      = sample["title"]
    audio_path = sample["audio_path"]
    dali_lines = sample["lines"]

    print(f"\n{'='*60}")
    print(f"[{sample_idx+1}/{len(dali_samples)}] {title}  (id: {song_id})")
    print(f"{'='*60}")

    # ── Skip if audio is unavailable ──────────────────────
    if audio_path is None or not os.path.isfile(audio_path):
        print("   ⚠️  No audio file found — skipping.")
        results_summary.append({"id": song_id, "title": title,
                                 "wer": None, "cer": None, "status": "no_audio"})
        continue

    # 1. Load audio ────────────────────────────────────────
    print("   Loading audio …")
    try:
        audio, sr = load_and_resample_audio(audio_path, TARGET_SAMPLE_RATE)
        duration  = len(audio) / sr
        print(f"   Duration: {duration:.1f}s")
    except Exception as e:
        print(f"   ❌  Audio load failed: {e}")
        results_summary.append({"id": song_id, "title": title,
                                 "wer": None, "cer": None, "status": "load_error"})
        continue

    # 2. Transcribe ────────────────────────────────────────
    print("   Transcribing …")
    all_segments = transcribe_audio(
        audio, sr,
        chunk_sec=CHUNK_DURATION_SEC,
        overlap_sec=OVERLAP_SEC,
        language=LANGUAGE,
        task=TASK
    )
    print(f"   Raw segments: {len(all_segments)}")

    # 3. Clean & deduplicate ───────────────────────────────
    cleaned = deduplicate_and_clean_segments(all_segments, time_tolerance=2.0)
    print(f"   After dedup:  {len(cleaned)}")

    # 4. Format lyrics ────────────────────────────────────
    lyrics_lines = format_lyrics(cleaned, dali_lines=dali_lines)

    # 5. Evaluate ─────────────────────────────────────────
    metrics = compute_metrics(lyrics_lines, dali_lines)
    print(f"   WER: {metrics['wer']}  |  CER: {metrics['cer']}")

    # 6. Save lyrics to disk ───────────────────────────────
    safe_title  = re.sub(r"[^\w\s-]", "", title).strip().replace(" ", "_")
    output_path = os.path.join(OUTPUT_DIR, f"{safe_title}_{song_id}.txt")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"Title : {title}\n")
        f.write(f"ID    : {song_id}\n")
        f.write(f"WER   : {metrics['wer']}\n")
        f.write(f"CER   : {metrics['cer']}\n")
        f.write("\n" + "─" * 50 + "\n\n")
        f.write("\n".join(lyrics_lines))

    print(f"   ✅  Saved → {output_path}")

    # Print a short preview
    print("\n   ── LYRICS PREVIEW ──")
    for line in lyrics_lines[:12]:
        print(f"   {line}")
    if len(lyrics_lines) > 12:
        print(f"   … ({len(lyrics_lines)} lines total)")

    results_summary.append({
        "id"     : song_id,
        "title"  : title,
        "wer"    : metrics["wer"],
        "cer"    : metrics["cer"],
        "status" : "ok"
    })

print(f"\n✅ Done! Processed {len(dali_samples)} sample(s).")

---
## Section 11 — Summary Report

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(results_summary)
print("\n📊 Results Summary")
print("=" * 60)
print(summary_df.to_string(index=False))

valid = summary_df[summary_df["status"] == "ok"]
if not valid.empty:
    avg_wer = valid["wer"].dropna().mean()
    avg_cer = valid["cer"].dropna().mean()
    print(f"\n   Average WER: {avg_wer:.4f}")
    print(f"   Average CER: {avg_cer:.4f}")

# Save summary CSV
summary_path = os.path.join(OUTPUT_DIR, "summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\n✅ Summary saved → {summary_path}")

---
## 🔧 Tuning Tips

| Problem | Solution |
|---|---|
| Lines too long / too joined | Lower `NEW_LINE_GAP_SEC` (e.g. 1.0) |
| Lines too fragmented | Raise `NEW_LINE_GAP_SEC` (e.g. 2.0) |
| Missing verse breaks | Lower `NEW_VERSE_GAP_SEC` (e.g. 2.5) |
| Too many verse breaks | Raise `NEW_VERSE_GAP_SEC` (e.g. 5.0) |
| Poor transcription accuracy | Use `openai/whisper-small` or `whisper-medium` |
| Out of memory | Lower `CHUNK_DURATION_SEC` to 20 |
| Non-English songs | Change `LANGUAGE` to the song's language code |

---

## 🚀 Next Steps

1. **Vocal isolation** — Run `Demucs` before Whisper for cleaner input
2. **LoRA fine-tuning** — Use the preprocessed dataset to fine-tune on singing audio
3. **Larger model** — Swap `whisper-base` → `whisper-small` / `whisper-medium` for better WER
4. **Word-level timestamps** — Use `whisper-timestamped` for finer line-break control
5. **Gradio demo** — Wrap this pipeline in a Gradio UI

---
*AutoLyrics — Even Semester Projects, IIT Guwahati Coding Club, 2026*